# Расчёт размера выборки перед экспериментом

Продакт пришёл с задачей:

> «Запускаем тест новой подачи цены. Хочу быть уверен, что мы поймаем даже эффект в **0.5 п.п.** по конверсии в заказ»

**п.п. = процентный пункт**: рост конверсии с 30.0% до 30.5% — это +0.5 п.п. (но +1.7% относительных). Не путай — от этого зависит весь расчёт.

Твоя задача в этом ноутбуке:
1. Посчитать базовую конверсию и темп набора аудитории на исторических данных.
2. Оценить нужный размер выборки **аналитической формулой**.
3. Перепроверить **симуляциями** (Monte Carlo).
4. Сверить с реальным размером аудитории и дать продакту честный ответ: реализуемо или нет, и какие есть варианты.

In [ ]:
# pip install clickhouse-connect pandas numpy scipy statsmodels
import clickhouse_connect
import numpy as np
import pandas as pd
from scipy import stats

ch = clickhouse_connect.get_client(
    host="localhost", port=8123, username="default", password="platform"
)

def q(sql: str) -> pd.DataFrame:
    return ch.query_df(sql)

q("SELECT min(event_date), max(event_date), count() FROM ab.events")

## Шаг 1. Базовые величины из истории

Тебе нужны две цифры по **аудитории будущего эксперимента** (тот же регион и фильтры, что будут в карточке):

- `p_base` — базовая конверсия `screen_view → order_confirm` на пользователя в день;
- темп набора: сколько **уникальных** пользователей аудитории появляется в данных за 1 / 7 / 14 дней.

Подумай: почему уников за 14 дней не в 14 раз больше, чем за 1 день? Как это повлияет на длительность теста?

In [ ]:
# TODO: SQL по ab.events за предпериод:
#  1) p_base: доля пользователей с order_confirm среди пользователей со screen_view
#  2) uniqExact(user_id) за окна 1 / 7 / 14 дней по твоему региону

p_base = ...          # TODO
users_per_day = ...   # TODO: примерный темп набора НОВЫХ уников в день к концу второй недели

## Шаг 2. Аналитический калькулятор

Для сравнения двух долей размер выборки **на группу**:

$$n = \frac{2 \, (z_{1-\alpha/2} + z_{1-\beta})^2 \; \bar{p}(1-\bar{p})}{\delta^2}$$

где $\delta$ — MDE в долях (0.5 п.п. = 0.005), $\bar{p}$ — средняя конверсия двух групп, $z$ — квантили нормального распределения.

Напоминание терминов:
- **alpha** — допустимая вероятность ложного срабатывания (обычно 0.05);
- **power = 1 − beta** — вероятность поймать эффект, если он есть (обычно 0.8);
- **MDE** — минимальный эффект, который хотим уметь детектировать.

In [ ]:
def required_sample_size(p_base: float, mde_pp: float, alpha: float = 0.05, power: float = 0.8) -> int:
    """Размер выборки НА ГРУППУ для z-теста двух долей."""
    # TODO: квантили возьми из stats.norm.ppf
    ...

# TODO: посчитай n для сетки MDE: 0.5, 1, 2, 3 п.п. и сведи в таблицу:
# MDE | n на группу | всего | дней набора при твоём темпе аудитории

## Шаг 3. Проверка симуляциями (Monte Carlo)

Формула выше верна для идеальной доли. Симуляции нужны там, где формулы нет или она врёт: ratio-метрики, тяжёлые хвосты, зависимые наблюдения.

Алгоритм оценки мощности:
1. Возьми пользователей из исторических данных с их фактическим исходом (была конверсия / нет).
2. Много раз (например, 2000): сэмплируй две группы по `n` пользователей, группе B искусственно «подними» конверсию на MDE, прогони стат-тест, запомни p-value.
3. Мощность = доля итераций, где p-value < alpha. Она должна сойтись с заявленной (0.8) на размере из формулы.

Бонус: проверь и alpha — прогони то же самое **без** инъекции эффекта. Доля ложных срабатываний должна быть ≈ 0.05.

In [ ]:
def simulate_power(user_outcomes: np.ndarray, n_per_group: int, mde_pp: float,
                   alpha: float = 0.05, n_sims: int = 2000, seed: int = 42) -> float:
    """Доля значимых результатов при истинном эффекте = mde_pp."""
    rng = np.random.default_rng(seed)
    significant = 0
    for _ in range(n_sims):
        # TODO: сэмплируй группы A и B из user_outcomes
        # TODO: инъекция эффекта в B: часть нулей переверни в единицы с нужной вероятностью
        # TODO: z-тест двух долей (statsmodels.stats.proportion.proportions_ztest)
        ...
    return significant / n_sims

# TODO: сравни мощность из симуляций с формулой на 2-3 значениях MDE

## Шаг 4. Сверка с реальностью и ответ продакту

Узнай фактический размер аудитории под твоими фильтрами — сплитовалка считает это сама:

In [ ]:
import requests

# Эксперимент должен быть уже создан через POST /experiments
EXPERIMENT_CODE = "pricing_point_estimate"
preview = requests.post(f"http://localhost:8000/experiments/{EXPERIMENT_CODE}/audience/preview").json()
preview

### Вопросы на самопроверку

1. Хватит ли аудитории на MDE 0.5 п.п. за разумный срок? Сколько недель понадобилось бы?
2. Какие у продакта есть варианты, если не хватает? Назови минимум три и издержки каждого.
3. Почему нельзя просто «подержать тест подольше, пока не станет значимо»?
4. Что изменится в расчёте, если целевой метрикой сделать `trips_per_user` (среднее), а не конверсию (долю)?

### Формат сдачи

Короткое сообщение продакту (3–6 предложений): реализуемо ли MDE 0.5 п.п., что предлагаешь
вместо этого и почему. Плюс таблица MDE → размер выборки → длительность из Шага 2.
Числа из расчёта перенеси в раздел 8 карточки эксперимента.